In [1]:
import os
import ray
from ray.rllib.algorithms.algorithm import Algorithm
from ray.tune.registry import register_env
import torch
# Custom code
from env.multi_agent_environment import LuxMultiAgentEnv
from base_model_config import get_base_model_config

# Reload packages without having to restart kernel
%load_ext autoreload
%autoreload 2

In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
random_seed = 42

In [4]:
rllib_base_algo_dir = os.path.abspath("../saved_algos_rllib/base_algo")
# rllib_base_algo_log_dir = os.path.abspath("../training_logs_rllib/base_algo")
# os.makedirs(rllib_base_algo_log_dir, exist_ok=True)

In [5]:
if ray.is_initialized():
    ray.shutdown()

ray.init(
    num_cpus=os.cpu_count(), # Or specify a number
    num_gpus=torch.cuda.device_count(), # Uncomment if using GPU
    ignore_reinit_error=True,
    log_to_driver=False # Set to True for easier debugging inline
)

2025-04-27 20:47:37,284	INFO worker.py:1843 -- Started a local Ray instance. View the dashboard at 127.0.0.1:8266 


Python version:,3.10.6
Ray version:,2.44.1
Dashboard:,http://127.0.0.1:8266


In [6]:
def create_base_multi_agent_env(config):
    return LuxMultiAgentEnv(config)
register_env("lux_base_multi_agent_env", create_base_multi_agent_env)

In [7]:
# Load or create algo.
base_model_config = get_base_model_config(
    random_seed=random_seed,
    num_env_runners=8,
    num_envs_per_env_runner=4,
    num_gpus=1
)

# Find the saved algo
if os.path.exists(rllib_base_algo_dir):
    try:
        print(f"Loading algorithm from {rllib_base_algo_dir}...")
        algo = Algorithm.from_checkpoint(rllib_base_algo_dir)
    except Exception as e:
        print(f"Error restoring from saved file {rllib_base_algo_dir}: {e}")
        print("Building new algorithm instead.")
        algo = base_model_config.build_algo()
else:
    print("No saved algo found. Starting new training run.")
    os.makedirs(rllib_base_algo_dir, exist_ok=True)
    algo = base_model_config.build_algo()

2025-04-27 20:47:43,192	WARNING deprecation.py:50 -- DeprecationWarning: `AlgorithmConfig.evaluation(evaluation_num_workers=..)` has been deprecated. Use `AlgorithmConfig.evaluation(evaluation_num_env_runners=..)` instead. This will raise an error in the future!
2025-04-27 20:47:43,195	WARNING algorithm_config.py:4704 -- You are running PPO on the new API stack! This is the new default behavior for this algorithm. If you don't want to use the new API stack, set `config.api_stack(enable_rl_module_and_learner=False,enable_env_runner_and_connector_v2=False)`. For a detailed migration guide, see here: https://docs.ray.io/en/master/rllib/new-api-stack-migration-guide.html


No saved algo found. Starting new training run.


C:\Users\karlh\AppData\Local\Programs\Python\Python310\lib\site-packages\ray\rllib\algorithms\algorithm.py:512: RayDeprecationWarning: This API is deprecated and may be removed in future Ray releases. You could suppress this warning by setting env variable PYTHONWARNINGS="ignore::DeprecationWarning"
`UnifiedLogger` will be removed in Ray 2.7.
  return UnifiedLogger(config, logdir, loggers=None)
C:\Users\karlh\AppData\Local\Programs\Python\Python310\lib\site-packages\ray\tune\logger\unified.py:53: RayDeprecationWarning: This API is deprecated and may be removed in future Ray releases. You could suppress this warning by setting env variable PYTHONWARNINGS="ignore::DeprecationWarning"
The `JsonLogger interface is deprecated in favor of the `ray.tune.json.JsonLoggerCallback` interface and will be removed in Ray 2.7.
  self._loggers.append(cls(self.config, self.logdir, self.trial))
C:\Users\karlh\AppData\Local\Programs\Python\Python310\lib\site-packages\ray\tune\logger\unified.py:53: RayDep

In [8]:
train_episodes = 100
checkpoint_freq = 10

training_metrics = []
print(f"Starting RLlib PPO (LSTM) training...")
print(f"Checkpoints will be saved to: {rllib_base_algo_dir}")

# Training Loop
for i in range(train_episodes):
    result = algo.train()

    # Extract metrics for easier access
    env_runners_metrics = result.get('env_runners', {})
    agent_returns = env_runners_metrics.get('agent_episode_returns_mean', {})
    learners_metrics = result.get('learners', {})
    p0_learner_metrics = learners_metrics.get('p0', {})
    p1_learner_metrics = learners_metrics.get('p1', {})

    print(f"Iteration: {result.get('training_iteration', i+1)} - "
          f"P_0_reward_mean:{agent_returns.get('player_0', float('nan'))}, "
          f"P_1_reward_mean:{agent_returns.get('player_1', float('nan'))}")

    # Store important metrics for this iteration
    metrics_to_store = {
        'iteration': result.get('training_iteration', i+1),
        'time_this_iter_s': result.get('time_this_iter_s', float('nan')),
        'time_total_s': result.get('time_total_s', float('nan')),

        # Sampling/Rollout Metrics
        'num_env_steps_sampled_this_iter': env_runners_metrics.get('num_env_steps_sampled', 0),
        'num_env_steps_sampled_lifetime': result.get('num_env_steps_sampled_lifetime', 0),
        'episode_return_mean_training': env_runners_metrics.get('episode_return_mean', float('nan')),
        'episode_len_mean_training': env_runners_metrics.get('episode_len_mean', float('nan')),
        'p0_reward_mean': agent_returns.get('player_0', float('nan')), # Assumes player_0 -> p0
        'p1_reward_mean': agent_returns.get('player_1', float('nan')), # Assumes player_1 -> p1

        # Learner Metrics (p0)
        'p0_total_loss': p0_learner_metrics.get('total_loss', float('nan')),
        'p0_policy_loss': p0_learner_metrics.get('policy_loss', float('nan')),
        'p0_vf_loss': p0_learner_metrics.get('vf_loss', float('nan')),
        'p0_entropy': p0_learner_metrics.get('entropy', float('nan')),
        'p0_kl_loss': p0_learner_metrics.get('mean_kl_loss', float('nan')),
        'p0_vf_explained_var': p0_learner_metrics.get('vf_explained_var', float('nan')),

        # Learner Metrics (p1)
        'p1_total_loss': p1_learner_metrics.get('total_loss', float('nan')),
        'p1_policy_loss': p1_learner_metrics.get('policy_loss', float('nan')),
        'p1_vf_loss': p1_learner_metrics.get('vf_loss', float('nan')),
        'p1_entropy': p1_learner_metrics.get('entropy', float('nan')),
        'p1_kl_loss': p1_learner_metrics.get('mean_kl_loss', float('nan')),
        'p1_vf_explained_var': p1_learner_metrics.get('vf_explained_var', float('nan')),
    }
    training_metrics.append(metrics_to_store)

    # Checkpointing
    if (i + 1) % checkpoint_freq == 0:
        algo.save_checkpoint(checkpoint_dir=rllib_base_algo_dir)
        print(f"Checkpoint saved in directory {rllib_base_algo_dir}")

print("Training finished.")

Starting RLlib PPO (LSTM) training...
Checkpoints will be saved to: C:\Users\karlh\GIT\uib\luxAI-s3\saved_algos_rllib\base_algo
Iteration: 1 - P_0_reward_mean:0.37788461538461543, P_1_reward_mean:0.0
Iteration: 2 - P_0_reward_mean:1.9634615384615388, P_1_reward_mean:0.04615384615384615
Iteration: 3 - P_0_reward_mean:0.4307692307692308, P_1_reward_mean:0.0
Iteration: 4 - P_0_reward_mean:3.4375, P_1_reward_mean:0.3846153846153847
Iteration: 5 - P_0_reward_mean:0.9326923076923078, P_1_reward_mean:0.23846153846153853
Iteration: 6 - P_0_reward_mean:1.5278846153846157, P_1_reward_mean:0.20384615384615382
Iteration: 7 - P_0_reward_mean:14.300961538461529, P_1_reward_mean:11.827884615384601
Iteration: 8 - P_0_reward_mean:2.897115384615385, P_1_reward_mean:0.24519230769230774
Iteration: 9 - P_0_reward_mean:12.50192307692306, P_1_reward_mean:6.480769230769226
Iteration: 10 - P_0_reward_mean:1.5817307692307692, P_1_reward_mean:0.004807692307692308
Checkpoint saved in directory C:\Users\karlh\GIT\

In [9]:
# Final Save
print(f"Saving final algo to directory: {rllib_base_algo_dir}")
algo.save_to_path(path=rllib_base_algo_dir)
print(f"Final checkpoint saved in directory {rllib_base_algo_dir}")

# Cleanup
algo.stop()
ray.shutdown()
print("RLlib Algorithm stopped and Ray shut down.")

Saving final algo to directory: C:\Users\karlh\GIT\uib\luxAI-s3\saved_algos_rllib\base_algo
Final checkpoint saved in directory C:\Users\karlh\GIT\uib\luxAI-s3\saved_algos_rllib\base_algo
Environment closed.
Environment closed.
Environment closed.
Environment closed.
Environment closed.
Environment closed.
Environment closed.
Environment closed.
RLlib Algorithm stopped and Ray shut down.


In [10]:
# Save and display metrics
import pandas as pd
metrics_df = pd.DataFrame(training_metrics)
metrics_df.to_csv("training_metrics.csv", index=False)
metrics_df

,iteration,time_this_iter_s,time_total_s,num_env_steps_sampled_this_iter,num_env_steps_sampled_lifetime,episode_return_mean_training,episode_len_mean_training,p0_reward_mean,p1_reward_mean,p0_total_loss,...,p0_vf_loss,p0_entropy,p0_kl_loss,p0_vf_explained_var,p1_total_loss,p1_policy_loss,p1_vf_loss,p1_entropy,p1_kl_loss,p1_vf_explained_var
0,1,31.605998,31.605998,16160,16160,0.377885,505.0,0.377885,0.000000,-0.148826,...,0.051551,68.215660,0.038884,0.399927,-0.142890,-0.150624,0.000009,68.376556,0.038626,-1.000000
1,2,32.068548,63.674546,16160,32320,2.009615,505.0,1.963462,0.046154,0.122506,...,0.177850,68.310791,0.028742,0.364138,-0.163803,-0.182165,0.013402,68.269424,0.016535,0.121890
2,3,28.570903,92.245449,16160,48480,0.430769,505.0,0.430769,0.000000,-0.600958,...,0.201206,68.154442,0.027217,0.120500,0.097980,0.092409,0.000181,68.102058,0.017968,-1.000000
3,4,30.403682,122.649131,16160,64640,3.822115,505.0,3.437500,0.384615,0.740699,...,0.925900,68.408775,0.010472,-0.008956,0.218831,0.209751,0.004277,68.160454,0.016013,-0.297561
4,5,33.908472,156.557604,16160,80800,1.171154,505.0,0.932692,0.238462,0.140092,...,0.247627,68.270401,0.011896,0.047664,-0.157618,-0.218660,0.056814,68.081734,0.014091,0.041972
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,34.500730,3098.221793,16160,1551360,1.972115,505.0,1.184615,0.787500,0.159643,...,0.142914,66.855865,0.008714,-0.000663,0.118645,-0.030562,0.136876,64.715492,0.008553,-0.015078
96,97,33.360475,3131.582268,16160,1567520,32.199038,505.0,21.536538,10.662500,2.539309,...,2.620629,67.047897,0.009942,0.003182,1.162752,0.152136,0.996684,64.272667,0.009664,0.026716
97,98,33.359971,3164.942239,16160,1583680,16.572115,505.0,12.539423,4.032692,1.158263,...,1.160250,67.186836,0.013213,0.070406,0.435815,0.084386,0.335234,64.764961,0.011233,0.022088
98,99,33.564571,3198.506810,16160,1599840,0.452885,505.0,0.452885,0.000000,0.180046,...,0.003208,66.803940,0.008149,-1.000000,0.308294,0.289009,0.000875,64.635704,0.012770,-1.000000
